In [ ]:
# Data Preparation Pipeline
This notebook consolidates functional connectivity (FC) matrices from three datasets (Neurocon, PPMI, TaoWu) into a unified format for downstream analysis.

**Purpose:**
- Load and standardize correlation matrices from multiple sources
- Create a master metadata CSV linking subjects to their FC matrices
- Ensure consistent labeling across datasets (PD vs Control)

# Import necessary libraries for file handling and data manipulation
import os
import pandas as pd
import numpy as np
import glob
from scipy.io import loadmat  # For loading MATLAB .mat files

# ===== DEFINE FILE PATHS =====
# Root directory containing all datasets
DATA_DIR = r'C:\Users\nikna\Documents\pp_datasets'

# Paths to metadata files (contain subject IDs and diagnoses)
metadata_path = os.path.join(DATA_DIR, 'Metadata_v5/Metadata_V5')

# Paths to each dataset's functional connectivity matrices
neurocon_path = os.path.join(DATA_DIR, 'neurocon')
ppmi_path = os.path.join(DATA_DIR, 'ppmi')
taowu_path = os.path.join(DATA_DIR, 'taoWu')

# Metadata CSV files for each dataset
metadata_files = ["Neurocon_metadata.csv", "PPMI_metadata.csv", "TaoWu_metadata.csv"]

# Target file suffix to identify correlation matrices (not timeseries)
targetfile_suffix = 'schaefer100_correlation_matrix'

# Output directory for processed data
output_dir = r'C:\Users\danie\Documents\Projects\Master thesis\Tara\Scripts\Outputs'

# Map each metadata file to its corresponding dataset folder
# This allows us to locate FC matrices for each subject
dataset_map = {
    "Neurocon_metadata.csv": neurocon_path,
    "PPMI_metadata.csv":     ppmi_path,
    "TaoWu_metadata.csv":    taowu_path,
}

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize empty list to accumulate metadata for all subjects
# Each entry will contain: subject_id, label, diagnosis, dataset_source, npy_path
metadata_list = []

# ===== MAIN PROCESSING LOOP =====
# Iterate through each dataset's metadata file
for metadata_file in metadata_files:
    print("processing metadata file:", metadata_file)
    metadata_file_path = os.path.join(metadata_path, metadata_file)
    
    # Check if metadata file exists
    if not os.path.exists(metadata_file_path):
        print("metadata file not found:", metadata_file_path)
        continue

    # Load metadata CSV (contains Subject IDs and Group labels)
    meta_df = pd.read_csv(metadata_file_path)
    id_col = "Subject"  # Column name for subject IDs
    label_col = "Group"  # Column name for diagnosis (PD/Control/etc.)
    
    # Get the dataset path for this metadata file
    dataset_path = dataset_map.get(metadata_file)
    print("using dataset folder:", dataset_path)
    
    # Extract unique subject IDs from metadata
    subject_ids = meta_df[id_col].astype(str).drop_duplicates().tolist()
    print("found", len(subject_ids), "unique subjects in", metadata_file)

    # ===== PROCESS EACH SUBJECT =====
    # Loop through all subject IDs to locate their FC matrices
    for sid in subject_ids:
        # Special handling for PPMI: folder names include diagnosis prefix
        # Example: "sub-patient123" for PD subjects, "sub-control456" for controls
        if metadata_file == "PPMI_metadata.csv":
            # Get the group/diagnosis for this subject
            group = meta_df.loc[meta_df[id_col].astype(str) == sid, label_col].values[0]
            # Convert to lowercase for folder naming
            group_lower = str(group).lower()
            # Replace 'pd' with 'patient' for folder naming convention
            if group_lower == 'pd':
                group_lower = 'patient'
            folder_name = f"sub-{group_lower}{sid}"
        else:
            # For Neurocon and TaoWu, use subject ID as-is
            folder_name = f"sub-{sid}"
        
        # Construct full path to subject's folder
        subject_folder = os.path.join(dataset_path, folder_name)

        # Check if subject folder exists
        if not os.path.isdir(subject_folder):
           print("folder not found for subject:", sid, "expected:", folder_name)
           continue
     
        # ===== LOCATE CORRELATION MATRIX FILE =====
        # Search for the Schaefer 100 correlation matrix (not timeseries)
        # Pattern: *_schaefer100_correlation_matrix.mat
        search_pattern = os.path.join(subject_folder, f"*{targetfile_suffix}.mat")
        matching_files = glob.glob(search_pattern)

        # Skip if no correlation matrix found
        if len(matching_files) == 0:
           print("no correlation matrix found for:", sid)
           continue

        # Use the first matching file (should only be one)
        FC_path = matching_files[0]

        # ===== LOAD AND SAVE CORRELATION MATRIX =====
        try:
            # Load MATLAB file containing the correlation matrix
            mat_data = loadmat(FC_path)
            
            # Extract the numerical matrix (ignore MATLAB metadata keys starting with '_')
            # MATLAB files contain metadata keys like '__header__', '__version__', etc.
            keys = [k for k in mat_data.keys() if not k.startswith('_')]
            FC_matrix = mat_data[keys[0]]
            
            # Convert to NumPy array (100x100 correlation matrix)
            FC_matrix = np.array(FC_matrix)

            # Get the diagnosis label for this subject from metadata
            diagnosis = meta_df.loc[meta_df[id_col].astype(str) == sid, label_col].values[0]
            
            # Create FC subfolder in output directory
            fc_output_dir = os.path.join(output_dir, 'FC')
            os.makedirs(fc_output_dir, exist_ok=True)
            
            # Create descriptive filename: dataset_diagnosis_id.npy
            # Example: "PPMI_pd_123.npy" or "Neurocon_control_456.npy"
            dataset_name = metadata_file.replace("_metadata.csv", "")
            diagnosis_clean = str(diagnosis).lower().replace(" ", "_")
            npy_filename = f"{dataset_name}_{diagnosis_clean}_{sid}.npy"
            npy_save_path = os.path.join(fc_output_dir, npy_filename)
            
            # Save correlation matrix as NumPy binary file (.npy)
            np.save(npy_save_path, FC_matrix)

            # ===== STANDARDIZE LABELS =====
            # Convert all diagnoses to binary labels:
            # 1 = Parkinson's Disease (PD, Prodromal, SWEDD)
            # 0 = Healthy Control
            diagnosis_upper = str(diagnosis).upper()
            binary_label = 1 if any(keyword in diagnosis_upper for keyword in ['PD', 'PRODROMAL', 'SWEDD']) else 0

            # Append subject information to metadata list
            metadata_list.append({
                "subject_id": sid,
                "label": binary_label,           # Binary label (0=Control, 1=PD)
                "diagnosis": diagnosis,          # Original diagnosis string
                "dataset_source": dataset_name,  # Which dataset this subject came from
                "npy_path": npy_save_path        # Path to saved .npy file
            })

        except Exception as e:
            print(f"Error processing subject {sid}: {e}")
            continue


# ===== CREATE MASTER METADATA FILE =====
# Convert accumulated metadata list to pandas DataFrame
master_df = pd.DataFrame(metadata_list)

# Remove duplicate subjects (keep first occurrence)
# This handles cases where a subject might appear multiple times
master_df = master_df.drop_duplicates(subset=['subject_id', 'dataset_source'], keep='first')

# Save master metadata CSV
# This file links all subjects to their FC matrices and labels
master_csv_path = os.path.join(output_dir, "master_metadata.csv")
master_df.to_csv(master_csv_path, index=False)

# Print summary statistics
print("Data preparation complete.")
print("Saved:", master_csv_path)
print("Total subjects processed:", len(metadata_list))
print("Unique subjects after deduplication:", len(master_df))

In [1]:
import os
import pandas as pd
import numpy as np
import glob
from scipy.io import loadmat

In [9]:
DATA_DIR = r'C:\Users\nikna\Documents\pp_datasets'

metadata_path = r'C:\Users\nikna\Documents\pp_datasets\Metadata_v5-20251021T115759Z-1-001\Metadata_v5\Metadata_v5'
neurocon_path = r'C:\Users\nikna\Documents\pp_datasets\neurocon-20251021T115804Z-1-001\neurocon\neurocon'
ppmi_path = r'C:\Users\nikna\Documents\pp_datasets\ppmi_v2-20251021T115808Z-1-001\ppmi_v2\ppmi'
taowu_path = r'C:\Users\nikna\Documents\pp_datasets\taowu-20251021T115812Z-1-001\taowu\taowu'

metadata_files = ["Neurocon_metadata.csv", "PPMI_metadata.csv", "TaoWu_metadata.csv"]
targetfile_suffix = 'schaefer100_correlation_matrix'
output_dir = r'C:\Users\nikna\Documents\pp_datasets\processed_data_2'

# mapping metadata files to the path for the dataset folders
dataset_map = {
    "Neurocon_metadata.csv": neurocon_path,
    "PPMI_metadata.csv":     ppmi_path,
    "TaoWu_metadata.csv":    taowu_path,
}

os.makedirs(output_dir, exist_ok=True)

# 3. Initialize an empty list to store metadata
#    - metadata_list = []

metadata_list = []

In [11]:
for metadata_file in metadata_files:
    print("processing metadata file:", metadata_file)
    metadata_file_path = os.path.join(metadata_path, metadata_file)
    
    if not os.path.exists(metadata_file_path):
        print("metadata file not found:", metadata_file_path)
        continue

    meta_df = pd.read_csv(metadata_file_path)
    id_col = "Subject"
    label_col = "Group" # ppmi runs into problem here
    #getting the dataset path for this metadata file
    dataset_path = dataset_map.get(metadata_file)
    print("using dataset folder:", dataset_path)
    meta_df = meta_df[~meta_df[label_col].astype(str).str.contains('Prodromal|SWEDD', case=False, na=False)]
    subject_ids = meta_df[id_col].astype(str).drop_duplicates().tolist()
    print("found", len(subject_ids), "unique subjects in", metadata_file)

# 5. Loop through all subject folders in that dataset
#    - (e.g., using os.listdir or glob.glob)
#    - Get the subject_id from the folder name.
#    - Get the diagnosis (PD or control) from the folder/file name.
    # looping throughsubject folders
    for sid in subject_ids:
        # Special handling for PPMI: combine Group + Subject
        if metadata_file == "PPMI_metadata.csv":
            # Get the group/diagnosis for this subject
            group = meta_df.loc[meta_df[id_col].astype(str) == sid, label_col].values[0]
            # Convert to lowercase
            group_lower = str(group).lower()
            # Replace 'pd' with 'patient' for folder naming
            if group_lower == 'pd':
                group_lower = 'patient'
            folder_name = f"sub-{group_lower}{sid}"
        else:
            # For other datasets, use subject ID as-is
            if str(sid).startswith('sub-'):
                folder_name = str(sid)
            else:
                folder_name = f"sub-{sid}"
        
        subject_folder = os.path.join(dataset_path, folder_name)

        if not os.path.isdir(subject_folder):
           print("folder not found for subject:", sid, "expected:", folder_name)
           continue
     
# 6. Find the target file
#    - Search inside the subject's folder for the file that ends with
#      TARGET_FILE_SUFFIX (and is a correlation_matrix, not timeseries).
#    - (e.g., sub-control032057_schaefer100_correlation_matrix)
    
        search_pattern = os.path.join(subject_folder, f"*{targetfile_suffix}.mat")
        matching_files = glob.glob(search_pattern)

        if len(matching_files) == 0:
           print("no correlation matrix found for:", sid)
           continue

        FC_path = matching_files[0]

# 7. If the target file is found:
#    - a. Load the matrix (e.g., using numpy.loadtxt(file_path)).
#    - b. Define a new, clean save path for this subject's data
#         (e.g., OUTPUT_DIR / f"{subject_id}_fc_matrix.npy").
#    - c. Save the loaded matrix as a .npy file (numpy.save).
#    - d. Append this subject's info to metadata_list:
#         {
#           'subject_id': subject_id,
#           'diagnosis': 'PD' or 'Control',
#           'dataset_source': dataset_name,
#           'npy_path': new_save_path
#        #         }
        try:
            mat_data = loadmat(FC_path)
            
            # Extracting the numerical matrix (and ignoring MATLAB metadata keys)
            # This line of code finds the key that contains the 100x100 matrix
            keys = [k for k in mat_data.keys() if not k.startswith('_')]
            FC_matrix = mat_data[keys[0]]
            
            # numpy conversion
            FC_matrix = np.array(FC_matrix)

            # Getting the diagnosis for each subject from meta_df
            diagnosis = meta_df.loc[meta_df[id_col].astype(str) == sid, label_col].values[0]
            
            # Create FC subfolder
            fc_output_dir = os.path.join(output_dir, 'FC')
            os.makedirs(fc_output_dir, exist_ok=True)
            
            # Create better filename: dataset_diagnosis_id.npy
            dataset_name = metadata_file.replace("_metadata.csv", "")
            diagnosis_clean = str(diagnosis).lower().replace(" ", "_")
            npy_filename = f"{dataset_name}_{diagnosis_clean}_{sid}.npy"
            npy_save_path = os.path.join(fc_output_dir, npy_filename)
            
            np.save(npy_save_path, FC_matrix)

            # Standardize label to 1 for PD/Prodromal/SWEDD and 0 for Control
            diagnosis_upper = str(diagnosis).upper()
            binary_label = 1 if any(keyword in diagnosis_upper for keyword in ['PD']) else 0

# 8. After all loops are finished:
#    - Convert metadata_list to a pandas DataFrame.
#    - Save this DataFrame to a CSV file (e.g., OUTPUT_DIR / 'metadata.csv').
#    - This CSV will be the "master file" for all other scripts.

            metadata_list.append({
                "subject_id": sid,
                "label": binary_label,
                "diagnosis": diagnosis,
                "dataset_source": dataset_name,
                "npy_path": npy_save_path
            })

        except Exception as e:
            print(f"Error processing subject {sid}: {e}")
            continue


# 9. Print a success message
#    - (e.g., "Data preparation complete. Processed X subjects.")

master_df = pd.DataFrame(metadata_list)

# Remove duplicate rows (keep first occurrence)
master_df = master_df.drop_duplicates(subset=['subject_id', 'dataset_source'], keep='first')

master_csv_path = os.path.join(output_dir, "master_metadata.csv")
master_df.to_csv(master_csv_path, index=False)

print("Data preparation complete.")
print("Saved:", master_csv_path)
print("Total subjects processed:", len(metadata_list))
print("Unique subjects after deduplication:", len(master_df))

processing metadata file: Neurocon_metadata.csv
using dataset folder: C:\Users\nikna\Documents\pp_datasets\neurocon-20251021T115804Z-1-001\neurocon\neurocon
found 41 unique subjects in Neurocon_metadata.csv
processing metadata file: PPMI_metadata.csv
using dataset folder: C:\Users\nikna\Documents\pp_datasets\ppmi_v2-20251021T115808Z-1-001\ppmi_v2\ppmi
found 128 unique subjects in PPMI_metadata.csv
folder not found for subject: control3351 expected: sub-controlcontrol3351
folder not found for subject: control3353 expected: sub-controlcontrol3353
folder not found for subject: control3361 expected: sub-controlcontrol3361
folder not found for subject: control3368 expected: sub-controlcontrol3368
folder not found for subject: control3369 expected: sub-controlcontrol3369
folder not found for subject: control3389 expected: sub-controlcontrol3389
folder not found for subject: control3390 expected: sub-controlcontrol3390
folder not found for subject: control3563 expected: sub-controlcontrol3563